# SETTINGS

In [ ]:
import os
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# CONFIGURATION

In [ ]:
project_dir = r"C:\cache\Youtube-ETL_Project"
processed_dir = os.path.join(project_dir, "data", "processed")
sql_dir = os.path.join(project_dir, "sql")
latest_cleaned_path = os.path.join(processed_dir, "latest_cleaned_path.txt")

if os.path.exists(latest_cleaned_path):
    with open(latest_cleaned_path, "r", encoding="utf-8") as f:
        cleaned_path = f.read().strip()
else:
    cleaned_files = [
        os.path.join(processed_dir, file_name)
        for file_name in os.listdir(processed_dir)
        if file_name.startswith("cleaned_youtube_") and file_name.endswith(".csv")
    ]
    if not cleaned_files:
        raise FileNotFoundError(f"No cleaned CSV files found in: {processed_dir}")
    cleaned_path = max(cleaned_files, key=os.path.getmtime)

# Các cột bắt buộc phải có để ghi vào bảng staging_video_raw
staging_columns = [
    "video_id", "video_title", "channel_id", "channel_name", "subscriber_count",
    "publish_date", "video_url", "duration", "duration_seconds", "duration_category",
    "view_count", "like_count", "comment_count", "tags", "category",
    "thumbnail_url", "collection_date",
]

print(f"Cleaned CSV: {cleaned_path}")

# KẾT NỐI MYSQL

In [ ]:
load_dotenv()

mysql_host = os.getenv("MYSQL_HOST")
mysql_user = os.getenv("MYSQL_USER")
mysql_password = os.getenv("MYSQL_PASSWORD")
mysql_database = os.getenv("MYSQL_DATABASE")
mysql_port = int(os.getenv("MYSQL_PORT", "3306"))

required_vars = {
    "MYSQL_HOST": mysql_host,
    "MYSQL_USER": mysql_user,
    "MYSQL_PASSWORD": mysql_password,
    "MYSQL_DATABASE": mysql_database,
}

missing_vars = [name for name, value in required_vars.items() if not value]
if missing_vars:
    raise ValueError(f"Missing required environment variables: {', '.join(missing_vars)}")

encoded_password = quote_plus(mysql_password)
database_url = f"mysql+pymysql://{mysql_user}:{encoded_password}@{mysql_host}:{mysql_port}/{mysql_database}?charset=utf8mb4"

engine = create_engine(database_url)
print(f"Connected to MySQL database: {mysql_database}")

# Lưu ý: KHÔNG tự tạo database/table ở đây -> phải chạy sql/create_tables.sql
# 1 lần trước đó (VD bằng DBeaver) để có sẵn dim_channel, dim_video,
# fact_video_metrics, staging_video_raw.

# ĐỌC CLEANED CSV

In [ ]:
if not os.path.exists(cleaned_path):
    raise FileNotFoundError(f"Cleaned CSV not found: {cleaned_path}")

df = pd.read_csv(cleaned_path)

missing_cols = [c for c in staging_columns if c not in df.columns]
if missing_cols:
    raise ValueError(f"Cleaned CSV thiếu cột bắt buộc cho staging: {missing_cols}")

print(f"Loaded cleaned CSV shape: {df.shape}")
display(df.head())

# LOAD VÀO STAGING

In [ ]:
# Ghi đè (replace) toàn bộ staging_video_raw -> OK vì đây chỉ là bảng đệm
# cho đúng lần chạy hiện tại, KHÔNG phải nơi lưu lịch sử (khác với dim/fact).
df[staging_columns].to_sql(
    name="staging_video_raw",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000,
)

with engine.begin() as conn:
    staging_count = conn.execute(text("SELECT COUNT(*) FROM staging_video_raw")).scalar_one()

print(f"Đã ghi {staging_count} dòng vào staging_video_raw")

# TÁCH DIM / FACT (chạy trong 1 TRANSACTION)

In [ ]:
# Đọc sql/merge_staging_to_dim_fact.sql và chạy toàn bộ trong CÙNG 1
# transaction (engine.begin() tự COMMIT nếu ok, tự ROLLBACK nếu có lỗi
# giữa chừng) -> đảm bảo 3 bước insert dim_channel/dim_video/fact
# không bao giờ bị dở dang.
merge_sql_path = os.path.join(sql_dir, "merge_staging_to_dim_fact.sql")
with open(merge_sql_path, "r", encoding="utf-8") as f:
    sql_script = f.read()

statements = [s.strip() for s in sql_script.split(";") if s.strip() and not s.strip().startswith("--")]

with engine.begin() as conn:
    for stmt in statements:
        conn.execute(text(stmt))

print(f"Đã chạy {len(statements)} câu SQL để tách staging -> dim/fact")

# RESULT

In [ ]:
with engine.begin() as conn:
    dim_video_count = conn.execute(text("SELECT COUNT(*) FROM dim_video")).scalar_one()
    fact_count = conn.execute(text("SELECT COUNT(*) FROM fact_video_metrics")).scalar_one()

print(f"✅ Hoàn tất. dim_video: {dim_video_count} video | fact_video_metrics: {fact_count} snapshot")